# Product Recognition on Store Shelves

**Exam Project:** Developed for the final exam by **Massimo Modesti** and **Federico Tampieri**.  
This notebook implements a system for recognizing cereal box products on supermarket shelves, following the exam requirements.

---

# Table of Contents  

[Setup and Configuration](#setup-and-configuration)  

0. [Contest](#Contest)
1. [Step A - Multiple Product Detection](#step-a---multiple-product-detection)
    1. [Description](#description)
    2. [Starting setup](#starting-setup)
    3. [Scene Preprocessing and Color Cue Extraction](#scene-preprocessing-and-color-cue-extraction)
    4. [Initialize SIFT and BFMatcher](#initialize-sift-and-bfmatcher)
    5. [Model Loading and Feature Preparation](#model-loading-and-feature-preparation)
2. [Step B Multiple Instance Detection](#step-b-multiple-instance-detection)
3. [Step C Extended Scenario](#step-c-extended-scenario)


# Setup and configuration
Before running the code in this project, we strongly recommend setting up a dedicated Python virtual environment.   
This helps to:
- install exactly the packages needed by this project,
- avoid version conflicts with other projects on your machine,
- ensure that the notebooks run in the same conditions we used during development.

To make this process easier, we provide a requirements.txt file. This file lists all the necessary libraries (and their versions) so that you can install them in one shot inside the virtual environment. Once the environment is created and activated, installing the dependencies is as simple as: `pip install -r requirements.txt`.   

By following this setup, you greatly reduce the risk of missing packages or incompatible versions, and you ensure that the code in the following sections can be executed smoothly.

# Contest

In this project we address the problem of **product recognition on supermarket shelves** using classical computer vision techniques. The goal is to design a system that, given an image of a store shelf, can automatically detect and recognize cereal box products.

More specifically, the system operates under the assumption that **one reference image is available for each product** that needs to be recognized. Using these model images, the system must analyse a shelf scene and identify all visible cereal boxes belonging to the known product classes. For every product type present in the scene, the expected output includes:

1. the **number of detected instances**,  
2. the **size of each instance**, expressed as the width and height (in pixels) of its enclosing bounding box, and  
3. the **position of each instance** in the image reference frame, given by the coordinates of the bounding box center (in pixels).

The dataset provided for the project is organised into two main folders:

- **Models**: contains one reference image per product class, which serves as the template for recognition.  
- **Scenes**: contains different shelf images, which represent the test scenarios where the algorithm must locate and classify the cereal boxes.

Starting from this material, we develop and progressively refine a detection pipeline capable of handling increasing levels of difficulty.


## Step A - Multiple Product Detection

### Description
In this task we implement a first complete pipeline for detecting a single instance of each product in relatively simple shelf images. The system is given a set of model images (one reference image for each cereal box) and a set of scene images depicting supermarket shelves. For each scene, the goal is to decide which products are present and, for each detected product, to estimate the position and size of its bounding box in image coordinates.   

The proposed solution follows a classical feature-based approach:   
1. First, each scene image is pre-processed using a **LAB color space conversion** and **CLAHE** (Contrast Limited Adaptive Histogram Equalization) on the L channel to enhance local contrast and make feature extraction more robust.
2. Then, **SIFT keypoints and descriptors** are computed both on the pre-processed scene and on all model images.
    a. For each model, descriptors are matched to the scene using a BFMatcher with L2 norm, and Lowe’s ratio test is applied to keep only reliable correspondences.
    b. If the number of good matches is sufficient, we estimate a homography between model and scene using RANSAC, and use it to project the four corners of the model image into the scene.

From the projected corners we derive a rotated bounding box via `cv2.minAreaRect`, obtaining the center, width, height, and orientation of the detected instance. Additional filters are applied to improve robustness: very small boxes are discarded based on a minimum area threshold relative to the scene size, and for a subset of visually similar products we apply a color-based check by comparing the mean Hue of the detected region with the mean Hue of the corresponding model. Finally, for each scene, the system reports the detected products together with the center coordinates and bounding box dimensions of every instance, and overlays the rotated bounding boxes and product IDs directly on the shelf image for visual inspection.


### Starting setup
In this section, we import necessary libraries, define directories for template and scene images, enumerate template IDs, set thresholds for matching and color filtering, and specify visualization parameters.

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

# Directories containing template (model) and scene (shelf) images
MODELS_DIR       = "./models/"    # contains files named 0.jpg, 1.jpg, ..., 26.jpg
SCENES_DIR       = './scenes/'    # contains files e1.png to e5.png

MODEL_IDS        = [0, 1, 11, 19, 24, 25, 26]
SCENE_FILES      = ["e1.png", "e2.png", "e3.png", "e4.png", "e5.png"]

# Models to be discriminated with color filter
CONFUSE_MODELS   = {1, 11, 0, 26}
HUE_DIFF_THRESH  = 17  # Allowed Hue difference (degrees)

# Matching and filtering parameters
MIN_MATCHES      = 30     # Minimum number of feature matches/inliers required for a valid detection
RATIO_TEST       = 0.7    # Lowe's ratio test parameter to filter ambiguous matches
MIN_AREA_RATIO   = 0.01   # Minimum bounding-box area ratio relative to scene size
RANSAC_THRESH    = 3.0    # RANSAC reprojection threshold (pixels)

# Visualization styles
BOX_COLOR    = (0, 255, 0)  # green box outlines
CENTER_COLOR = (0, 0, 255)  # red centers
TEXT_COLOR   = (0, 255, 0)  # green text
FONT         = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE   = 0.7
TEXT_THICK   = 2
BOX_THICK    = 10
CIRCLE_RAD   = 4

### Scene Preprocessing and Color Cue Extraction

Now, we add two functions:  
1. `def preprocess_scene(img_scene)`: Converts the image to LAB and applies CLAHE on the L (luminance) channel to boost local contrast without amplifying noise, then converts back to BGR—yielding clearer gradients and textures for more reliable feature detection;
2. `def calc_mean_hue(img_bgr)`: Converts the image to HSV and returns the average Hue (0–180 in OpenCV), providing a compact descriptor of the dominant color used later to disambiguate visually similar packages after geometric matching.

In [ ]:
def preprocess_scene(img_scene): 
    lab = cv2.cvtColor(img_scene, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    lab = cv2.merge((cl, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def calc_mean_hue(img_bgr): 
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    return float(np.mean(hsv[:, :, 0]))

### Initialize SIFT and BFMatcher


In [ ]:
sift = cv2.SIFT_create()
bf   = cv2.BFMatcher(cv2.NORM_L2)

### Model Loading and Feature Preparation

This block loads all reference models and prepares them for matching.
For each `MODEL_ID`, it reads the model image, computes SIFT keypoints and descriptors, records its size, and (for visually similar items listed in `CONFUSE_MODELS`) caches the model’s mean Hue for later color-based disambiguation.
All metadata are stored in the `models` dictionary, which serves as the lookup table used during scene detection.

In [ ]:
# Model loading and preprocessing
models = {} 
for mid in MODEL_IDS: 
    path = os.path.join(MODELS_DIR, f"{mid}.jpg")
    img_model = cv2.imread(path)
    if img_model is None:
        raise FileNotFoundError(f"Model image not found: {path}")

    # keypoints and descriptors
    kp_model, des_model = sift.detectAndCompute(img_model, None)
    h_model, w_model = img_model.shape[:2]

    # mean hue for filtering models
    mean_hue = None
    if mid in CONFUSE_MODELS:
        mean_hue = calc_mean_hue(img_model)

    models[mid] = {
        'img': img_model,
        'kp': kp_model,
        'des': des_model,
        'size': (w_model, h_model),
        'mean_hue': mean_hue
    }

### Scene-Level Detection Pipeline: Matching, Geometric Verification, and Visualization

In this code block we process each shelf scene and run the complete detection pipeline against all reference models. We start by iterating over the list of scene filenames (SCENE_FILES). For each scene_file, we load the corresponding image from the SCENES_DIR. If the image is missing, we immediately raise a FileNotFoundError, so that errors in the dataset are caught early instead of silently propagating.

Once the scene image is loaded, we apply our pre-processing function preprocess_scene. As explained earlier, this step converts the image to LAB, applies CLAHE to the L channel, and converts back to BGR. We do this to improve local contrast and make SIFT feature extraction more robust in the presence of lighting variations. We also store the height and width of the processed scene, since we will later use them to filter out detections whose area is too small relative to the image size.

We then extract SIFT keypoints and descriptors from the pre-processed scene by calling sift.detectAndCompute. This gives us a set of local features that we will match against each model. For debugging and inspection purposes, we print the number of keypoints and the descriptor shape, so we can quickly see whether the feature extraction step behaves as expected on each scene. We also initialize an empty dictionary detections, which will be used to store all valid detections for that scene, grouped by model ID.

After this setup, we loop over all pre-computed models stored in the models dictionary. For each model ID (mid), we retrieve its SIFT keypoints, descriptors, and image size. If either the model or the scene has no descriptors, we cannot perform matching reliably, so we skip that model. Otherwise, we execute the core of our geometric matching pipeline:

SIFT matching and Lowe’s ratio test.
We use bf.knnMatch with k=2 to find, for each model descriptor, the two closest scene descriptors. Then we apply Lowe’s ratio test, keeping only those matches where the best distance is significantly smaller than the second best. This reduces ambiguous matches and improves robustness. If the number of “good” matches is below a threshold (MIN_MATCHES), we consider that there is not enough evidence to support a detection and skip that model for the current scene.

Homography estimation with RANSAC.
From the surviving matches, we extract the corresponding 2D coordinates in the model (src_pts) and in the scene (dst_pts). We then estimate a homography using cv2.findHomography with the RANSAC flag. RANSAC allows us to discard outlier matches and obtain a robust transformation between model and scene. If no homography is found or the number of inliers (matches consistent with the transformation) is again below MIN_MATCHES, we treat the detection as unreliable and skip it.

Projection of model corners.
Assuming a valid homography, we define the four corners of the model image in its own coordinate system and project them onto the scene using cv2.perspectiveTransform. This gives the location of the model’s outline in the scene.

Rotated bounding box.
From the projected corner points, we compute the minimum-area bounding rectangle using cv2.minAreaRect. This provides the center coordinates (cx, cy), the width and height of the box, and its rotation angle. We also derive the four vertices (box_pts) for later visualization.

Minimum area filter.
We want to discard detections that are too small to be meaningful, as they are often due to noise or accidental matches. To this end, we compute the area of the bounding box and compare it with a fraction (MIN_AREA_RATIO) of the whole scene area. If the detection is too small, we skip it.

Optional color-based filter.
For certain models that are visually similar (e.g. same layout but different dominant color), geometry alone is not sufficient. For those models listed in CONFUSE_MODELS, we add a color consistency check: we compute an inverse perspective transform to crop the region of interest (roi) in the scene corresponding to the model rectangle, compute its mean Hue using calc_mean_hue, and compare it with the pre-computed mean Hue of the model. If the difference exceeds a threshold (HUE_DIFF_THRESH), we reject the detection as a likely false positive.

If a detection passes all these tests, we store it in the detections dictionary under the corresponding model ID, saving the center, size, angle, and box vertices. This allows us to keep track of all accepted detections for the current scene.

After processing all models, we print a textual summary for the scene: for each product ID, we report how many instances were found and the geometric details (position, width, height, angle) of each one. Finally, we visualize the detections by drawing the rotated bounding boxes, centroids, and model IDs on a copy of the original scene image (vis). We use Matplotlib to display the annotated image inline in the notebook, converting from BGR to RGB so that colors appear correctly. This visual feedback is crucial both to validate the correctness of our pipeline and to qualitatively inspect its behavior across different scenes.  

In [ ]:
# Process each scene image
for scene_file in SCENE_FILES:
    # Load the scene image
    scene_path = os.path.join(SCENES_DIR, scene_file)
    img_scene = cv2.imread(scene_path)
    if img_scene is None:
        raise FileNotFoundError(f"Scene image not found: {scene_path}") 

    # Pre-process the scene (e.g. resizing, normalization)
    img_proc = preprocess_scene(img_scene)
    h_scene, w_scene = img_proc.shape[:2]

    # Extract SIFT keypoints and descriptors from the scene
    kp_scene, des_scene = sift.detectAndCompute(img_proc, None)
    print(f"\n=== Analisi scena {scene_file} ===")
    print(f"Kp scena: {len(kp_scene)}  Descriptors: {None if des_scene is None else des_scene.shape}")

    # Dictionary to store found detections (key: model_id, value: list of detections)
    detections = {} 
    
    # Loop through each model to search in the scene
    for mid, data in models.items(): 
        # Retrieve keypoints, descriptors and model dimensions
        kp_model = data['kp']
        des_model = data['des']
        w_model, h_model = data['size']

        print(f"\n[Model {mid}]: kp_model={len(kp_model)}  des_model={None if des_model is None else des_model.shape}")
        
        # Check that descriptors exist for both model and scene
        if des_model is None or des_scene is None:
            print("  => Skip: descriptors missing")
            continue

        # ========== 1) SIFT MATCHING + LOWE RATIO TEST ==========
        # Find the k=2 best matches for each model descriptor
        matches = bf.knnMatch(des_model, des_scene, k=2) 
        print(f"  Raw matches: {len(matches)}")
        
        # Apply Lowe's ratio test: keep only matches with distance significantly lower
        # than the second best match (reduces ambiguity)
        good = [m for m,n in matches if m.distance < RATIO_TEST * n.distance]
        print(f"  Good matches (ratio<{RATIO_TEST}): {len(good)}")
        
        # If good matches are too few, skip this model
        if len(good) < MIN_MATCHES:
            print(f"  => Skip: too few good matches (<{MIN_MATCHES})")
            continue

        # ========== 2) HOMOGRAPHY ESTIMATION WITH RANSAC ==========
        # Extract coordinates of corresponding keypoints
        # src_pts: points in model (query), dst_pts: points in scene (train)
        src_pts = np.float32([kp_model[m.queryIdx].pt for m in good]).reshape(-1,1,2)
        dst_pts = np.float32([kp_scene[m.trainIdx].pt for m in good]).reshape(-1,1,2)
        
        # Calculate homography matrix (perspective transformation) with RANSAC
        # RANSAC filters outliers for robust estimation
        M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, RANSAC_THRESH)
        
        if M is None:
            print("  => Skip: homography not found")
            continue
        
        # Count inliers (points that respect the transformation)
        inliers = int(mask.sum())
        print(f"  Inliers RANSAC: {inliers}")
        
        # If inliers are too few, the match is not reliable
        if inliers < MIN_MATCHES:
            print(f"  => Skip: few inliers (<{MIN_MATCHES})")
            continue

        # ========== 3) TRANSFORM MODEL CORNERS ==========
        # Define the 4 corners of the model (rectangle in model coordinates)
        corners = np.float32([[0,0],[w_model,0],[w_model,h_model],[0,h_model]]).reshape(-1,1,2)
        
        # Apply perspective transformation to find where they are projected in the scene
        dst_c  = cv2.perspectiveTransform(corners, M)
        pts    = dst_c.reshape(-1,2).astype(np.float32)

        # ========== 4) CALCULATE ROTATED BOUNDING BOX ==========
        # Calculate the minimum oriented rectangle enclosing the transformed points
        rot_rect = cv2.minAreaRect(pts)
        (cx, cy), (w_box, h_box), angle = rot_rect  # center, dimensions, rotation angle
        box_pts = cv2.boxPoints(rot_rect).astype(int)  # 4 vertices of rotated box

        # ========== 5) MINIMUM AREA FILTER ==========
        # Discard detections with area too small relative to the scene
        # (possible false positives or irrelevant detections)
        if w_box * h_box < MIN_AREA_RATIO * (w_scene * h_scene):
            print(f"  => Skip: area too small (<{MIN_AREA_RATIO*100:.2f}% scene)")
            continue

        # ========== 6) CONDITIONAL COLOR FILTER ==========
        # For easily confusable models (e.g. same design, different color)
        # verify mean color correspondence (hue in HSV)
        if mid in CONFUSE_MODELS:
            # Define source rectangle (model) and destination (scene)
            src_rect = np.float32([[0,0],[w_model,0],[w_model,h_model],[0,h_model]])
            dst_rect = np.float32(box_pts)
            
            # Calculate inverse transformation to extract ROI from scene
            M_inv    = cv2.getPerspectiveTransform(dst_rect, src_rect)
            roi      = cv2.warpPerspective(img_scene, M_inv, (w_model, h_model))
            
            # Calculate mean hue of ROI and compare with model hue
            mean_h_roi = calc_mean_hue(roi)
            diff_h     = abs(mean_h_roi - data['mean_hue'])
            print(f"  Hue ROI={mean_h_roi:.1f}, Model={data['mean_hue']:.1f}, Δ={diff_h:.1f}")
            
            # If color difference exceeds threshold, discard the detection
            if diff_h > HUE_DIFF_THRESH:
                print(f"  => Skip: color difference too high (>±{HUE_DIFF_THRESH}°)")
                continue

        # ========== VALID DETECTION - SAVE ==========
        # Store detection data for this model
        detections.setdefault(mid, []).append({
            'center': (int(round(cx)), int(round(cy))),  # bbox center
            'width':  int(round(w_box)),                  # width
            'height': int(round(h_box)),                  # height
            'angle':  angle,                              # rotation angle
            'box_pts': box_pts                            # 4 vertices of box
        })
        print(f"  *** Detected model instance {mid} ({inliers} inliers) ***")

    # ========== RESULTS REPORT ==========
    print(f"\nResults for {scene_file}:")
    if not detections:
        print("  No product recognized.")
    
    # Print details for each detected model
    for pid, dets in detections.items():
        print(f"  Model {pid} - {len(dets)} instance(s) found:")
        for idx, det in enumerate(dets, 1):
            cx, cy = det['center']
            w_b, h_b = det['width'], det['height']
            print(f"    Instance {idx} {{position: ({cx},{cy}), w={w_b}px, h={h_b}px, angle={det['angle']:.1f}°}}")

    # ========== VISUALIZATION ==========
    # Create copy of image to draw results
    vis = img_scene.copy()
    
    # Draw each detection
    for pid, dets in detections.items():
        for det in dets:
            # Draw the rotated bounding box
            cv2.drawContours(vis, [det['box_pts']], 0, BOX_COLOR, BOX_THICK)
            
            # Draw the centroid
            cv2.circle(vis, det['center'], CIRCLE_RAD, CENTER_COLOR, -1)
            
            # Add text with model ID, centered on detection
            text = f"ID: {pid}"
            (tw, th), _ = cv2.getTextSize(text, FONT, FONT_SCALE, TEXT_THICK)
            tx = det['center'][0] - tw // 2  # center horizontally
            ty = det['center'][1] + th // 2  # center vertically
            cv2.putText(vis, text, (tx, ty), FONT, FONT_SCALE, TEXT_COLOR, TEXT_THICK)

    # Display results with Matplotlib (for notebooks/IDEs)
    plt.figure(figsize=(10, 8))
    plt.title(f"Detections — {scene_file}")
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))  # Convert BGR -> RGB for correct colors
    plt.axis("off")
    plt.show()